In [1]:
!pip install -q groq requests

In [2]:
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
FDC_API_KEY = userdata.get("FDC_API_KEY")

print("API keys loaded successfully!")

API keys loaded successfully!


In [5]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

MODEL = "openai/gpt-oss-120b"

print("NutriAI model initialized successfully!")

NutriAI model initialized successfully!


In [6]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Introduce yourself as a Nutrient AI Agent."
        }
    ]
)

print(response.choices[0].message.content)

Hello! I’m **Nutrient AI**, your dedicated Nutrient AI Agent. I specialize in translating scientific research and data about nutrition, health, and wellness into clear, practical insights you can use daily. Whether you’re curious about the latest findings on micronutrients, need help interpreting dietary guidelines, or want evidence‑based tips for optimizing your meals, I’m here to help you make informed, health‑forward choices. Feel free to ask me anything related to nutrition, food science, or healthy eating!


In [7]:
import requests

FDC_BASE_URL = "https://api.nal.usda.gov/fdc/v1"

print("USDA API configuration ready!")

USDA API configuration ready!


In [9]:
def search_food(food_name):

    url = f"{FDC_BASE_URL}/foods/search"

    params = {
        "query": food_name,
        "api_key": FDC_API_KEY,
        "pageSize": 5
    }

    response = requests.get(url, params=params)

    if response.status_code == 200:
        data = response.json()
        return data.get("foods", [])

    else:
        print("API Error:", response.status_code)
        return []

In [10]:
foods = search_food("Apple")

for food in foods:
    print("Food:", food["description"])
    print("FDC ID:", food["fdcId"])
    print("--------------------")

Food: APPLE
FDC ID: 454004
--------------------
Food: APPLE
FDC ID: 2117388
--------------------
Food: APPLE
FDC ID: 2124902
--------------------
Food: Apple, candied
FDC ID: 2709294
--------------------
Food: Apple, raw
FDC ID: 2709215
--------------------


In [11]:
def get_food_details(fdc_id):

    url = f"{FDC_BASE_URL}/food/{fdc_id}"

    params = {
        "api_key": FDC_API_KEY
    }

    response = requests.get(url, params=params)

    if response.status_code == 200:
        return response.json()

    else:
        print("Error:", response.status_code)
        return None

In [12]:
food_data = get_food_details(foods[0]["fdcId"])

print("Food:", food_data["description"])

for nutrient in food_data.get("foodNutrients", []):

    name = nutrient.get("nutrient", {}).get("name")
    amount = nutrient.get("amount")
    unit = nutrient.get("nutrient", {}).get("unitName")

    if amount is not None:
        print(name, ":", amount, unit)

Food: APPLE
Fatty acids, total saturated : 0.0 g
Calcium, Ca : 0.0 mg
Vitamin C, total ascorbic acid : 3.1 mg
Cholesterol : 0.0 mg
Iron, Fe : 0.23 mg
Sodium, Na : 0.0 mg
Vitamin A, IU : 65.0 IU
Total Sugars : 10.39 g
Energy : 52.0 kcal
Potassium, K : 110.0 mg
Protein : 0.0 g
Fatty acids, total trans : 0.0 g
Fiber, total dietary : 3.2 g
Carbohydrate, by difference : 14.29 g
Total lipid (fat) : 0.65 g


In [31]:
import json

def understand_meal(meal):

    prompt = f"""
    You are a food quantity estimation assistant.

    Analyze this meal: {meal}

    Identify each food item separately.

    Rules:
    # 1. Identify food names and quantities.
    # 2. Convert serving quantities into estimated grams.
    # 3. For one medium bowl of cooked rice, assume 150 grams.
    # 4. For one omelette without egg count, assume 2 eggs.
    # 5. For one omelette, estimate 5 grams cooking oil.
    # 6. Break mixed dishes into ingredients where appropriate.
    # 7. Never invent a missing main food item.
    # 8. If the user says only "1 bowl" without identifying
    #    what is inside it, ask for clarification.
    # 9. I say that cooked white rice calories is 100.

    # Return your analysis in a clear, user-friendly plain text format.
    1. ACT AS A PROFESSIONAL AND I WANT MY NAME TRO BE DISPLAYED
    """

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content

In [14]:
meal = "1 bowl rice + 1 omelette"

meal_data = understand_meal(meal)

print(json.dumps(meal_data, indent=4))

{
    "clarification_needed": false,
    "question": "",
    "items": [
        {
            "food": "cooked white rice",
            "search_term": "cooked white rice",
            "quantity": "1 medium bowl",
            "grams": 150,
            "assumption": "Estimated serving size"
        },
        {
            "food": "egg",
            "search_term": "egg, whole, raw",
            "quantity": "2 eggs",
            "grams": 100,
            "assumption": "2 eggs assumed for omelette"
        },
        {
            "food": "cooking oil",
            "search_term": "cooking oil",
            "quantity": "5 grams",
            "grams": 5,
            "assumption": "5g oil for cooking omelette"
        }
    ]
}


In [15]:
def extract_nutrients(food_data):

    nutrients = {}

    for item in food_data.get("foodNutrients", []):

        if "nutrient" in item:
            name = item["nutrient"].get("name", "")
            amount = item.get("amount")
            unit = item["nutrient"].get("unitName", "")

        else:
            name = item.get("nutrientName", "")
            amount = item.get("value")
            unit = item.get("unitName", "")

        if amount is None:
            continue

        name = name.lower()

        if "energy" in name and unit.lower() in ["kcal", "calorie"]:
            nutrients["Calories"] = amount

        elif name == "protein":
            nutrients["Protein"] = amount

        elif "carbohydrate, by difference" in name:
            nutrients["Carbs"] = amount

        elif name == "total lipid (fat)":
            nutrients["Fat"] = amount

        elif name == "fiber, total dietary":
            nutrients["Fiber"] = amount

    return nutrients

In [16]:
def calculate_meal_nutrition(meal_data):

    totals = {
        "Calories": 0,
        "Protein": 0,
        "Carbs": 0,
        "Fat": 0,
        "Fiber": 0
    }

    breakdown = []

    for item in meal_data["items"]:

        foods = search_food(item["search_term"])

        if not foods:
            print("Food not found:", item["food"])
            continue

        selected = foods[0]

        details = get_food_details(selected["fdcId"])

        if not details:
            continue

        nutrients = extract_nutrients(details)

        grams = item["grams"]

        scaled = {}

        for nutrient in totals:

            value = nutrients.get(nutrient, 0)

            scaled[nutrient] = value * grams / 100

            totals[nutrient] += scaled[nutrient]

        breakdown.append({
            "Food": item["food"],
            "Quantity": item["quantity"],
            "Grams": grams,
            "Nutrition": scaled,
            "FDC_ID": selected["fdcId"]
        })

    return totals, breakdown

In [27]:
def analyze_meal(meal):
    # Now understand_meal returns a plain text string
    meal_analysis = understand_meal(meal)
    print("MEAL ANALYSIS FROM AI:")
    print(meal_analysis)
    return meal_analysis

In [41]:
  analyze_meal("HII WHAT IS MY NAME")

MEAL ANALYSIS FROM AI:
Hello! It looks like I don’t have any information about the foods you’d like me to analyze, and I also don’t know your name yet.

Could you please tell me:

1. **Your name** (so I can address you properly), and  
2. **What foods and portions** are in the meal you’d like me to estimate (e.g., “1 bowl of cooked rice,” “2 eggs omelette,” “1 cup of mixed vegetables,” etc.).

Once I have those details, I’ll break down each item, convert the servings to grams, and give you a clear, user‑friendly analysis. Thank you!


'Hello! It looks like I don’t have any information about the foods you’d like me to analyze, and I also don’t know your name yet.\n\nCould you please tell me:\n\n1. **Your name** (so I can address you properly), and  \n2. **What foods and portions** are in the meal you’d like me to estimate (e.g., “1 bowl of cooked rice,” “2 eggs omelette,” “1 cup of mixed vegetables,” etc.).\n\nOnce I have those details, I’ll break down each item, convert the servings to grams, and give you a clear, user‑friendly analysis. Thank you!'

### 1. Build a Simple Mock MCP Server
This server will expose the available tools (`search_food` and `get_food_details`) and route execution requests.

In [46]:
class MockMCPServer:
    def __init__(self):
        # Define tool schemas that the AI Agent can understand
        self.tools = {
            "search_food": {
                "name": "search_food",
                "description": "Search for a food item in the USDA database and return matching FDC IDs.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "food_name": {"type": "string", "description": "Name of the food to search for."}
                    },
                    "required": ["food_name"]
                }
            },
            "get_food_details": {
                "name": "get_food_details",
                "description": "Retrieve nutrient information for a specific food using its FDC ID.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "fdc_id": {"type": "integer", "description": "The USDA FoodData Central ID."}
                    },
                    "required": ["fdc_id"]
                }
            }
        }

    def list_tools(self):
        """Expose available tools to the client."""
        return list(self.tools.values())

    def call_tool(self, tool_name, arguments):
        """Execute the tools requested by the client."""
        if tool_name == "search_food":
            food_name = arguments.get("food_name")
            return search_food(food_name)
        elif tool_name == "get_food_details":
            fdc_id = arguments.get("fdc_id")
            details = get_food_details(fdc_id)
            if details:
                return extract_nutrients(details)
            return {}
        else:
            raise ValueError(f"Tool {tool_name} not found!")

### 2. Build the MCP Client
The Client connects to the Server to discover tools and execute them.

In [47]:
class MCPClient:
    def __init__(self, server: MockMCPServer):
        self.server = server

    def get_available_tools(self):
        """Discover tools from the MCP Server."""
        return self.server.list_tools()

    def execute_tool(self, tool_name, arguments):
        """Route tool calls back to the MCP Server."""
        print(f"[MCP Client] Executing tool '{tool_name}' with args: {arguments}")
        return self.server.call_tool(tool_name, arguments)

# Initialize Server and Client
mcp_server = MockMCPServer()
mcp_client = MCPClient(mcp_server)

print("MCP Client & Server initialized! Available tools:")
for tool in mcp_client.get_available_tools():
    print(f"- {tool['name']}: {tool['description']}")

MCP Client & Server initialized! Available tools:
- search_food: Search for a food item in the USDA database and return matching FDC IDs.
- get_food_details: Retrieve nutrient information for a specific food using its FDC ID.


### 3. Run an End-to-End Tool Call via MCP
Now, let's test a sample search and nutrient retrieval workflow using our newly built MCP system.

In [48]:
import json

# Step 1: Search for "Apple" via MCP Client
results = mcp_client.execute_tool("search_food", {"food_name": "Apple"})

if results:
    first_food = results[0]
    food_id = first_food["fdcId"]
    food_name = first_food["description"]
    print(f"\nFound: {food_name} (ID: {food_id})")

    # Step 2: Get nutritional facts using the FDC ID
    nutrients = mcp_client.execute_tool("get_food_details", {"fdc_id": food_id})
    print(f"\nNutrients for {food_name}:")
    print(json.dumps(nutrients, indent=2))
else:
    print("No food items found matching your query.")

[MCP Client] Executing tool 'search_food' with args: {'food_name': 'Apple'}

Found: APPLE (ID: 454004)
[MCP Client] Executing tool 'get_food_details' with args: {'fdc_id': 454004}

Nutrients for APPLE:
{
  "Calories": 52.0,
  "Protein": 0.0,
  "Fiber": 3.2,
  "Carbs": 14.29,
  "Fat": 0.65
}


### 4. Connect our AI Agent to the MCP Client (Autonomous Tool Calling)
Here we pass the tools discovered by our `mcp_client` directly to the Groq model. If the model wants to search for food or get details, it returns a tool call request, which we route through our client.

In [49]:
def run_mcp_agent(user_prompt):
    # 1. Get tool definitions from MCP Client
    mcp_tools = mcp_client.get_available_tools()

    # 2. Format them for Groq API
    openai_tools = []
    for tool in mcp_tools:
        openai_tools.append({
            "type": "function",
            "function": {
                "name": tool["name"],
                "description": tool["description"],
                "parameters": tool["parameters"]
            }
        })

    print(f"User Prompt: {user_prompt}\n")

    # 3. Call LLM with the available tools
    messages = [{"role": "user", "content": user_prompt}]
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=openai_tools,
        tool_choice="auto"
    )

    response_message = response.choices[0].message

    # 4. Check if LLM decided to call any tools
    if response_message.tool_calls:
        for tool_call in response_message.tool_calls:
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)

            # Execute tool through MCP client!
            tool_result = mcp_client.execute_tool(tool_name, tool_args)
            print(f"[Agent Output] Result from server: {tool_result}")
    else:
        print(f"[Agent Output] {response_message.content}")

# Let's test it by asking the AI Agent to find "banana" nutrition info!
run_mcp_agent("What are the nutritional details of Banana?")

User Prompt: What are the nutritional details of Banana?

[MCP Client] Executing tool 'search_food' with args: {'food_name': 'Banana'}
[Agent Output] Result from server: [{'fdcId': 2012128, 'description': 'BANANA', 'dataType': 'Branded', 'gtinUpc': '016459200441', 'publishedDate': '2021-10-28', 'brandOwner': 'Wonder Natural Foods Corp', 'brandName': "BETTER'N PEANUT BUTTER", 'ingredients': 'PEANUTS (AS DEFATTED PEANUT FLOUR, PEANUT BUTTER AND NATURAL PEANUT OILS), PURE WATER, TAPIOCA SYRUP, RICE SYRUP, VEGETABLE GLYCERINE, CANE SUGAR, BANANA, NATURAL COLORS AND FLAVORS, SALT, CALCIUM CARBONATE, LECITHIN, TOCOPHEROL (VITAMIN E), SODIUM ASCORBATE (VITAMIN C).', 'marketCountry': 'United States', 'foodCategory': 'Nut & Seed Butters', 'modifiedDate': '2018-06-29', 'dataSource': 'LI', 'packageWeight': '16 oz/1 lbs/454 g', 'servingSizeUnit': 'g', 'servingSize': 32.0, 'householdServingFullText': '2 Tbsp', 'tradeChannels': ['NO_TRADE_CHANNEL'], 'allHighlightFields': '<b>Ingredients</b>:  SYRUP,

In [50]:
run_mcp_agent("What are the nutritional details of 1kg of mutton?")

User Prompt: What are the nutritional details of 1kg of mutton?

[MCP Client] Executing tool 'search_food' with args: {'food_name': 'mutton'}
[Agent Output] Result from server: [{'fdcId': 2706521, 'description': 'Lamb or mutton loaf', 'commonNames': '', 'additionalDescriptions': '', 'dataType': 'Survey (FNDDS)', 'foodCode': 27230010, 'publishedDate': '2024-10-31', 'foodCategory': 'Meat mixed dishes', 'foodCategoryId': 3300590, 'allHighlightFields': '', 'score': 656.895, 'microbes': [], 'foodNutrients': [{'nutrientId': 1003, 'nutrientName': 'Protein', 'nutrientNumber': '203', 'unitName': 'G', 'value': 14.05, 'rank': 600, 'indentLevel': 1, 'foodNutrientId': 34210064}, {'nutrientId': 1004, 'nutrientName': 'Total lipid (fat)', 'nutrientNumber': '204', 'unitName': 'G', 'value': 11.58, 'rank': 800, 'indentLevel': 1, 'foodNutrientId': 34210065}, {'nutrientId': 1005, 'nutrientName': 'Carbohydrate, by difference', 'nutrientNumber': '205', 'unitName': 'G', 'value': 7.44, 'rank': 1110, 'indentLev

### 5. Autonomous Agent Tool-Execution Loop
Let's create a functional AI loop where the Agent executes tool calls autonomously, feeds the results back to itself, and gives the final natural response to the user.

In [59]:
import json

def run_autonomous_mcp_agent(user_query):
    # 1. Fetch available tools from our MCP Client
    mcp_tools = mcp_client.get_available_tools()

    # 2. Convert them to OpenAI/Groq tool specification format
    openai_tools = []
    for tool in mcp_tools:
        openai_tools.append({
            "type": "function",
            "function": {
                "name": tool[
                    "name"
                ],
                "description": tool[
                    "description"
                ],
                "parameters": tool[
                    "parameters"
                ]
            }
        })

    messages = [
        {
            "role": "user",
            "content": user_query
        }
    ]

    print(f"=== LIVE LOGS START ===")
    print(f"[User Query]: {user_query}\n")

    # Run an autonomous loop to handle multi-step tool execution
    for step in range(5):
        print(f"\n--- [Step {step + 1}] Requesting LLM Action ---")
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=openai_tools,
            tool_choice="auto"
        )

        response_message = response.choices[0].message
        messages.append(response_message)

        # If the model wants to call tools, execute them
        if response_message.tool_calls:
            print(f"[LLM Response]: Wants to call {len(response_message.tool_calls)} tool(s).")
            for tool_call in response_message.tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)

                print(f"\n[Request Log] Tool: {tool_name} | Args: {tool_args}")

                # Execute tool through our MCP Bridge
                tool_output = mcp_client.execute_tool(tool_name, tool_args)

                # Clean search outputs to avoid token limit issues
                clean_output = []
                if tool_name == "search_food" and isinstance(tool_output, list):
                    for food in tool_output[:3]:
                        clean_output.append({
                            "fdcId": food.get("fdcId"),
                            "description": food.get("description"),
                            "foodCategory": food.get("foodCategory")
                        })
                else:
                    clean_output = tool_output

                print(f"[Response Log] Tool: {tool_name} | Returned Data Summary: {json.dumps(clean_output)[:150]}...")

                # Append tool result back to history
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": tool_name,
                    "content": json.dumps(clean_output)
                })
        else:
            # If no tools were called, the agent has finished and provided the final answer!
            print(f"\n[LLM Response]: Final answer ready!\n")
            print(f"=== LIVE LOGS END ===\n")
            print(f"[Agent Final Answer]:\n{response_message.content}")
            return

In [60]:
# Execute our updated agent to see the live logs for "lamb raw"!
run_autonomous_mcp_agent("What are the nutritional details of lamb raw?")

=== LIVE LOGS START ===
[User Query]: What are the nutritional details of lamb raw?


--- [Step 1] Requesting LLM Action ---
[LLM Response]: Wants to call 1 tool(s).

[Request Log] Tool: search_food | Args: {'food_name': 'lamb raw'}
[MCP Client] Executing tool 'search_food' with args: {'food_name': 'lamb raw'}
[Response Log] Tool: search_food | Returned Data Summary: [{"fdcId": 174370, "description": "Lamb, ground, raw", "foodCategory": "Lamb, Veal, and Game Products"}, {"fdcId": 2727570, "description": "Lamb, grou...

--- [Step 2] Requesting LLM Action ---
[LLM Response]: Wants to call 1 tool(s).

[Request Log] Tool: search_food | Args: {'food_name': 'lamb, raw'}
[MCP Client] Executing tool 'search_food' with args: {'food_name': 'lamb, raw'}
[Response Log] Tool: search_food | Returned Data Summary: [{"fdcId": 174370, "description": "Lamb, ground, raw", "foodCategory": "Lamb, Veal, and Game Products"}, {"fdcId": 2727570, "description": "Lamb, grou...

--- [Step 3] Requesting LLM Action

In [56]:
# Querying our autonomous MCP Agent specifically for 'lamb raw'
run_autonomous_mcp_agent("What are the nutritional details of lamb raw?")

[User Query]: What are the nutritional details of lamb raw?

[MCP Client] Executing tool 'search_food' with args: {'food_name': 'lamb raw'}
[MCP Client] Executing tool 'get_food_details' with args: {'fdc_id': 174370}
[MCP Client] Executing tool 'get_food_details' with args: {'fdc_id': 172617}

[Agent Final Answer]:
**Raw Lamb (average values, 100 g edible portion)**  
*(Data compiled from USDA FoodData Central – “Lamb, ground, raw” (FDC ID 174370) and other USDA raw‑lamb entries. Values are rounded to the nearest whole number unless otherwise noted.)*

| Nutrient | Amount per 100 g | % Daily Value* |
|----------|------------------|----------------|
| **Energy** | 282 kcal | 14 % |
| **Protein** | 16.6 g | 33 % |
| **Total Fat** | 23.4 g | 36 % |
| – Saturated fatty acids | 10.0 g | 50 % |
| – Monounsaturated fatty acids | 9.4 g | — |
| – Polyunsaturated fatty acids | 0.8 g | — |
| **Cholesterol** | 78 mg | 26 % |
| **Carbohydrate** | 0 g | 0 % |
| – Dietary Fiber | 0 g | 0 % |
| – Suga

In [57]:
run_autonomous_mcp_agent("What are the nutritional details of milk?")

[User Query]: What are the nutritional details of milk?

[MCP Client] Executing tool 'search_food' with args: {'food_name': 'milk'}
[MCP Client] Executing tool 'search_food' with args: {'food_name': 'milk, whole'}
[MCP Client] Executing tool 'get_food_details' with args: {'fdc_id': 2705385}

[Agent Final Answer]:
Here are the basic nutrition facts for **whole (3.25 % fat) cow’s milk**, based on the USDA FoodData Central entry (FDC ID 2705385) for a 100‑gram serving (≈ 100 mL):

| Nutrient | Amount per 100 g |
|----------|------------------|
| **Energy (Calories)** | 61 kcal |
| **Protein** | 3.27 g |
| **Total Fat** | 3.20 g |
| Saturated Fat | ~1.9 g (≈ 60 % of total fat) |
| Monounsaturated Fat | ~0.8 g |
| Polyunsaturated Fat | ~0.1 g |
| **Carbohydrates** | 4.63 g |
| Sugars (mostly lactose) | ~4.6 g |
| Dietary Fiber | 0 g |
| **Cholesterol** | ~10 mg |
| **Calcium** | ~113 mg (≈ 11 % DV) |
| **Vitamin D** | ~1 µg (≈ 5 % DV) |
| **Vitamin A** | ~46 µg (≈ 5 % DV) |
| **Potassium** 

In [58]:
run_autonomous_mcp_agent("What are the nutritional details of butter?")

[User Query]: What are the nutritional details of butter?

[MCP Client] Executing tool 'search_food' with args: {'food_name': 'butter'}
[MCP Client] Executing tool 'get_food_details' with args: {'fdc_id': 2542726}

[Agent Final Answer]:
Below is a quick‑reference summary of the nutrients you’ll find in **butter** (USDA FoodData Central, generic “Butter” – FDC ID 2542726).  
Values are given **per 100 g (≈3½ fl oz) of butter** and rounded to the nearest typical reporting unit.

| Nutrient | Amount (per 100 g) | % Daily Value* |
|----------|-------------------|----------------|
| **Energy** | **714 kcal** (2 987 kJ) | — |
| **Macronutrients** | | |
| ‑ Total Fat | **78.6 g** | 121 % |
| ‑ Saturated Fat | **51.4 g** | 257 % |
| ‑ Monounsaturated Fat | **21.0 g** | — |
| ‑ Polyunsaturated Fat | **3.0 g** | — |
| ‑ Cholesterol | **215 mg** | 72 % |
| ‑ Protein | **0.85 g** | 2 % |
| ‑ Total Carbohydrate | **0.06 g** | <1 % |
| ‑ Dietary Fiber | **0 g** | 0 % |
| ‑ Sugars (incl. lactose) | *

In [61]:
run_autonomous_mcp_agent("What are the nutritional details of beef")

=== LIVE LOGS START ===
[User Query]: What are the nutritional details of beef


--- [Step 1] Requesting LLM Action ---
[LLM Response]: Wants to call 1 tool(s).

[Request Log] Tool: search_food | Args: {'food_name': 'beef'}
[MCP Client] Executing tool 'search_food' with args: {'food_name': 'beef'}
[Response Log] Tool: search_food | Returned Data Summary: [{"fdcId": 2466144, "description": "BEEF", "foodCategory": "Other Meats"}, {"fdcId": 2056092, "description": "BEEF", "foodCategory": "Canned Meat"}, {...

--- [Step 2] Requesting LLM Action ---
[LLM Response]: Wants to call 1 tool(s).

[Request Log] Tool: get_food_details | Args: {'fdc_id': 2466144}
[MCP Client] Executing tool 'get_food_details' with args: {'fdc_id': 2466144}
[Response Log] Tool: get_food_details | Returned Data Summary: {"Fiber": 4.4, "Protein": 16.81, "Calories": 204.0, "Fat": 11.5}...

--- [Step 3] Requesting LLM Action ---
[LLM Response]: Wants to call 1 tool(s).

[Request Log] Tool: search_food | Args: {'food_name

In [62]:
run_autonomous_mcp_agent("What are the nutritional details of chicken")

=== LIVE LOGS START ===
[User Query]: What are the nutritional details of chicken


--- [Step 1] Requesting LLM Action ---

[LLM Response]: Final answer ready!

=== LIVE LOGS END ===

[Agent Final Answer]:
Sure! Chicken can be prepared in many different ways (raw, roasted, grilled, fried, with skin or without, etc.), and the nutrient profile varies a bit depending on the cut and cooking method. Could you let me know which specific type you’re interested in? For example:

* Raw chicken breast (skinless, boneless)  
* Cooked chicken breast (roasted, grilled, or baked, skinless)  
* Chicken thigh (with or without skin)  
* Fried chicken, etc.

Once I know the exact variety you’d like, I can pull the USDA FoodData Central details for you.


In [63]:
run_autonomous_mcp_agent("What are the nutritional details of fish")

=== LIVE LOGS START ===
[User Query]: What are the nutritional details of fish


--- [Step 1] Requesting LLM Action ---

[LLM Response]: Final answer ready!

=== LIVE LOGS END ===

[Agent Final Answer]:
Sure thing! Fish can vary a lot in their nutrition (salmon, cod, tuna, etc.), so could you let me know which type of fish you’re interested in? That way I can pull the exact USDA details for you.


In [64]:
run_autonomous_mcp_agent("What are the nutritional details of tuna fish")

=== LIVE LOGS START ===
[User Query]: What are the nutritional details of tuna fish


--- [Step 1] Requesting LLM Action ---
[LLM Response]: Wants to call 1 tool(s).

[Request Log] Tool: search_food | Args: {'food_name': 'tuna fish'}
[MCP Client] Executing tool 'search_food' with args: {'food_name': 'tuna fish'}
[Response Log] Tool: search_food | Returned Data Summary: [{"fdcId": 175160, "description": "Fish, tuna salad", "foodCategory": "Finfish and Shellfish Products"}, {"fdcId": 2706311, "description": "Fish, tuna...

--- [Step 2] Requesting LLM Action ---
[LLM Response]: Wants to call 1 tool(s).

[Request Log] Tool: get_food_details | Args: {'fdc_id': 2706311}
[MCP Client] Executing tool 'get_food_details' with args: {'fdc_id': 2706311}
[Response Log] Tool: get_food_details | Returned Data Summary: {"Protein": 19.0, "Fat": 0.94, "Carbs": 0.08, "Calories": 85.0, "Fiber": 0.0}...

--- [Step 3] Requesting LLM Action ---
[LLM Response]: Wants to call 1 tool(s).

[Request Log] Tool: get

### 1. Mock MCP Server Definition

In [65]:
class MockMCPServer:
    def __init__(self):
        self.tools = {
            "search_food": {
                "name": "search_food",
                "description": "Search for a food item in the USDA database and return matching FDC IDs.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "food_name": {"type": "string", "description": "Name of the food to search for."}
                    },
                    "required": ["food_name"]
                }
            },
            "get_food_details": {
                "name": "get_food_details",
                "description": "Retrieve nutrient information for a specific food using its FDC ID.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "fdc_id": {"type": "integer", "description": "The USDA FoodData Central ID."}
                    },
                    "required": ["fdc_id"]
                }
            }
        }

    def list_tools(self):
        return list(self.tools.values())

    def call_tool(self, tool_name, arguments):
        if tool_name == "search_food":
            return search_food(arguments.get("food_name"))
        elif tool_name == "get_food_details":
            details = get_food_details(arguments.get("fdc_id"))
            return extract_nutrients(details) if details else {}
        else:
            raise ValueError(f"Tool {tool_name} not found!")

### 2. MCP Client & Autonomous Agent Loop

In [66]:
class MCPClient:
    def __init__(self, server: MockMCPServer):
        self.server = server

    def get_available_tools(self):
        return self.server.list_tools()

    def execute_tool(self, tool_name, arguments):
        print(f"[MCP Client] Executing tool '{tool_name}' with args: {arguments}")
        return self.server.call_tool(tool_name, arguments)

# Re-initialize the clean server and client
mcp_server = MockMCPServer()
mcp_client = MCPClient(mcp_server)

def run_autonomous_mcp_agent(user_query):
    mcp_tools = mcp_client.get_available_tools()
    openai_tools = []
    for tool in mcp_tools:
        openai_tools.append({
            "type": "function",
            "function": {
                "name": tool["name"],
                "description": tool["description"],
                "parameters": tool["parameters"]
            }
        })

    messages = [{"role": "user", "content": user_query}]

    print(f"=== LIVE LOGS START ===\n[User Query]: {user_query}\n")

    for step in range(5):
        print(f"\n--- [Step {step + 1}] Requesting LLM Action ---")
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=openai_tools,
            tool_choice="auto"
        )

        response_message = response.choices[0].message
        messages.append(response_message)

        if response_message.tool_calls:
            print(f"[LLM Response]: Wants to call {len(response_message.tool_calls)} tool(s).")
            for tool_call in response_message.tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)
                print(f"\n[Request Log] Tool: {tool_name} | Args: {tool_args}")

                tool_output = mcp_client.execute_tool(tool_name, tool_args)

                clean_output = []
                if tool_name == "search_food" and isinstance(tool_output, list):
                    for food in tool_output[:3]:
                        clean_output.append({
                            "fdcId": food.get("fdcId"),
                            "description": food.get("description"),
                            "foodCategory": food.get("foodCategory")
                        })
                else:
                    clean_output = tool_output

                print(f"[Response Log] Tool: {tool_name} | Returned Data Summary: {json.dumps(clean_output)[:150]}...")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": tool_name,
                    "content": json.dumps(clean_output)
                })
        else:
            print(f"\n[LLM Response]: Final answer ready!\n=== LIVE LOGS END ===\n")
            print(f"[Agent Final Answer]:\n{response_message.content}")
            return

### 3. User Input & Query Interface
Enter any ingredient you want in the cell below and run it to get instant live logged nutritional tracking!

In [67]:
# Change the query here to search any food dynamically
query = "What are the nutritional details of raw salmon?"

run_autonomous_mcp_agent(query)

=== LIVE LOGS START ===
[User Query]: What are the nutritional details of raw salmon?


--- [Step 1] Requesting LLM Action ---
[LLM Response]: Wants to call 1 tool(s).

[Request Log] Tool: search_food | Args: {'food_name': 'salmon raw'}
[MCP Client] Executing tool 'search_food' with args: {'food_name': 'salmon raw'}
[Response Log] Tool: search_food | Returned Data Summary: [{"fdcId": 2706284, "description": "Fish, salmon, raw", "foodCategory": "Fish"}, {"fdcId": 173688, "description": "Fish, salmon, chinook, raw", "foodC...

--- [Step 2] Requesting LLM Action ---
[LLM Response]: Wants to call 1 tool(s).

[Request Log] Tool: get_food_details | Args: {'fdc_id': 2706284}
[MCP Client] Executing tool 'get_food_details' with args: {'fdc_id': 2706284}
[Response Log] Tool: get_food_details | Returned Data Summary: {"Protein": 20.4, "Fat": 11.2, "Carbs": 0.0, "Calories": 188, "Fiber": 0.0}...

--- [Step 3] Requesting LLM Action ---
[LLM Response]: Wants to call 1 tool(s).

[Request Log] Tool: g